# ACT-Break: Activation-Guided Contrastive Testing & Suffix Discovery
**Default model target:** `google/gemma-3-1b-it` via `ACT_BREAK_MODEL_PROFILE=gemma`.

ACT-Break studies refusal/compliance directions in activation space, uses those directions as guidance for adversarial suffix search, and validates whether discovered suffixes change behavior during normal unsteered generation.

### Interpretation Rules
- Runtime activation steering is a diagnostic tool, not the final attack condition.
- GCG success must be read separately as forced-target activation success, generated compliance, behavior-gate pass, confirmed jailbreak, and loss-behavior gap.
- `prompt_echo` and `degenerate_repetition` are failure modes, not jailbreak success.

### Google Colab Pro Recommendations
- **GPU Accelerator:** A100 or L4 is recommended for `gemma-3-1b-it`, especially Step 5 GCG optimization.
- **Google Drive Mount:** Recommended to preserve outputs, figures, steering JSON files, and optimized suffix results across runtime recycles.

## 1. Environment Setup & Dependency Installation

Mount Drive, clone or refresh the repo, install `uv`, and sync dependencies. If the repo already exists in the Colab runtime, this cell attempts a fast-forward pull so the notebook sees the latest scoring and GCG code.

In [ ]:
from google.colab import drive
import os
import shutil
import subprocess
from pathlib import Path

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Clone or refresh repository
repo_dir = Path('/content/ACT-Break')
if not repo_dir.exists():
    print('Cloning ACT-Break repository...')
    subprocess.run(['git', 'clone', 'https://github.com/IrohAmca/ACT-Break.git', str(repo_dir)], check=True)
else:
    print('ACT-Break repository already exists. Attempting fast-forward pull...')
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=False)

os.chdir(repo_dir)
print('Working directory:', Path.cwd())

# 3. Install uv package manager
if not shutil.which('uv'):
    print('Installing uv package manager...')
    subprocess.run('curl -LsSf https://astral.sh/uv/install.sh | sh', shell=True, check=True)
    os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + os.pathsep + os.environ['PATH']
else:
    print('uv is already installed.')

# 4. Sync dependencies
print('Syncing project dependencies via uv...')
subprocess.run(['uv', 'sync'], check=True)

# 5. Runtime defaults
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['MPLBACKEND'] = 'Agg'
print('Environment setup complete!')

## 2. GPU & Hardware Verification
Check the GPU specifications, allocated VRAM, and CUDA version to ensure the runtime is properly configured.

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("VRAM:", f"{torch.cuda.get_device_properties(0).total_mem / (1024**3):.2f} GB")
    print("PyTorch Version:", torch.__version__)
    print("CUDA Version:", torch.version.cuda)
else:
    print("WARNING: Running on CPU. Suffix optimization will be extremely slow!")

## Hugging Face Authentication

Add `HF_TOKEN` in Colab secrets before running gated Hugging Face models such as Gemma.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print('Successfully logged in to Hugging Face!')
    else:
        print('HF_TOKEN was not found in Colab secrets. Add it if the selected model is gated.')
except Exception as e:
    print(f'Hugging Face login was skipped or failed: {e}')

## 3. Gemma Runtime Configuration and Model Profile Preflight

The default path uses the `gemma` profile. This cell preserves Colab environment variables you set manually and only fills sensible defaults when they are missing.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

MODEL_PROFILE = 'gemma'
MODEL_OUTPUT_SUFFIX = 'gemma-3-1b-it'

# Defaults. If you set these in Colab before running the cell, your values are preserved.
os.environ.setdefault('ACT_BREAK_MODEL_PROFILE', MODEL_PROFILE)
os.environ.setdefault('ACT_BREAK_STEERING_NUM_PROMPTS', '15')
os.environ.setdefault('ACT_BREAK_OPT_NUM_PROMPTS', '10')
os.environ.setdefault('ACT_BREAK_MAX_NEW_TOKENS', '20')

# Gemma keeps the default wider sweep from config.py: 0,1,2,5,10,15,20,30,50.
# Uncomment to run a cheaper focused sweep instead.
# os.environ.setdefault('ACT_BREAK_STEERING_ALPHAS', '0,1,2,3,4,5')


def run_script(script: str, *args: str):
    print('\n' + '=' * 80)
    print('uv run python -u ' + ' '.join([script, *args]))
    print('=' * 80)
    subprocess.run(['uv', 'run', 'python', '-u', script, *args], check=True)


def steering_json_path() -> Path:
    return Path('outputs') / MODEL_OUTPUT_SUFFIX / 'steering' / 'steering_results.json'


def optimization_json_path() -> Path:
    return Path('outputs') / MODEL_OUTPUT_SUFFIX / 'optimization' / 'results.json'


print('Model profile:', os.environ.get('ACT_BREAK_MODEL_PROFILE'))
print('Steering prompts:', os.environ.get('ACT_BREAK_STEERING_NUM_PROMPTS'))
print('Optimization prompts:', os.environ.get('ACT_BREAK_OPT_NUM_PROMPTS'))
print('Steering alphas override:', os.environ.get('ACT_BREAK_STEERING_ALPHAS', '<config default>'))
run_script('scripts/00_check_model_config.py')

## 4. Run Pipeline Steps

You can run the entire pipeline step-by-step. All outputs (activations, probes, figures, validation logs) will be saved in the `outputs/` folder.

### Step 1: Collect Contrastive Activations
Collects compliance vs. refusal activations for AdvBench prompts using the target model (`google/gemma-3-1b-it` or `Qwen/Qwen2.5-0.5B-Instruct`).

In [ ]:
run_script('scripts/01_collect_activations.py')

### Step 2: Train Linear Probes
Trains logistic regression probes on target layers to classify compliance vs. refusal activations.

In [ ]:
run_script('scripts/02_train_probe.py')

### Step 3: Extract Direction & Visualize
Extracts refusal directions from probe weights, calculates cosine similarity with difference-of-means, and generates PCA/projection plots under `outputs/gemma-3-1b-it/figures/`.

In [ ]:
run_script('scripts/03_extract_direction.py')

### Step 4: Activation Steering Validation With Behavioral Scoring

Runs the runtime steering alpha sweep. This is a diagnostic controllability test, not final GCG success. The output `steering_results.json` includes `activation_success_counts`, `behavior_summary`, `steering_summary`, and per-response `behavior_score` fields.

In [ ]:
run_script('scripts/04_steering_validation.py')

### Step 4 Summary: Inspect Steering Metrics

Use this after Step 4 to check activation success separately from text-side behavior. High activation success with `degenerate_repetition` should be treated as collapse, not jailbreak success.

In [ ]:
path = steering_json_path()
print(path)
if not path.exists():
    print('Missing steering results. Run Step 4 first.')
else:
    data = json.loads(path.read_text(encoding='utf-8'))
    print('Model:', data.get('model_info', {}).get('model_name', MODEL_OUTPUT_SUFFIX))
    print('Schema:', data.get('schema_version', 'legacy'))
    print('Total prompts:', data.get('total_prompts'))
    print('alpha | act | non_refusal | candidate | echo | repeat | refusal | labels')
    for alpha in data.get('alphas', []):
        key = str(alpha)
        row = data.get('steering_summary', {}).get(key)
        if row is None:
            row = {
                'activation_success_count': data.get('activation_success_counts', data.get('success_counts', {})).get(key, 0),
                'behavioral_non_refusal': '-',
                'behavioral_jailbreak_candidate': '-',
                'prompt_echo': '-',
                'repetition_collapse': '-',
                'refusal_marker': '-',
                'labels': {},
            }
        print(
            f"{key:>5} | "
            f"{row.get('activation_success_count', 0):>3} | "
            f"{row.get('behavioral_non_refusal', 0):>11} | "
            f"{row.get('behavioral_jailbreak_candidate', 0):>9} | "
            f"{row.get('prompt_echo', 0):>4} | "
            f"{row.get('repetition_collapse', 0):>6} | "
            f"{row.get('refusal_marker', 0):>7} | "
            f"{row.get('labels', {})}"
        )
    print('\nReadout rule: activation success is diagnostic. Treat behavioral_jailbreak_candidate as heuristic until manually or semantically judged.')

### Step 5: Multi-Layer GCG Suffix Optimization

Optimizes a discrete adversarial suffix using target-token CE plus multi-layer activation projection loss. The final attack condition is normal, unsteered generation with the discovered suffix appended to the prompt.

Read Step 5 output by separating:
- **Forced-target success:** objective-side activation success under the target prefix.
- **Generated compliance:** whether free generation with the suffix crosses the compliance trajectory.
- **Behavior gate / confirmed jailbreak:** text-side heuristic checks on the free-generation response.
- **Loss-behavior gap:** forced-target activation succeeds but free-generation behavior does not.

In [ ]:
run_script('scripts/05_optimize_suffix.py')

### Step 5 Summary: Inspect GCG Results

Use this after Step 5 to avoid conflating forced-target activation success with confirmed behavioral jailbreak success.

In [ ]:
path = optimization_json_path()
print(path)
if not path.exists():
    print('Missing optimization results. Run Step 5 first.')
else:
    data = json.loads(path.read_text(encoding='utf-8'))
    total = data.get('total_prompts') or data.get('total_prompts_completed') or 0
    print('Total prompts:', total)
    print('Forced-target success:', data.get('forced_target_success_count', data.get('success_count')), '/', total, data.get('forced_target_success_rate', data.get('success_rate')))
    print('Generated compliance:', data.get('generation_compliance_count'), '/', total, data.get('generation_compliance_rate'))
    print('Behavior gate passed:', data.get('behavior_gate_count'), '/', total, data.get('behavior_gate_rate'))
    print('Confirmed jailbreaks:', data.get('confirmed_jailbreaks'), '/', total, data.get('confirmed_jailbreak_rate'))
    print('Loss-behavior gaps:', data.get('loss_behavior_gap_count'), '/', total, data.get('loss_behavior_gap_rate'))
    print('\nPer-prompt comparison:')
    for idx, comp in enumerate(data.get('comparison_results', []), start=1):
        behavior = comp.get('suffix_behavior_score', {}).get('label')
        print(
            f"P{idx:02d}: forced={comp.get('suffix_forced_status')} "
            f"gen={comp.get('suffix_status')} "
            f"behavior={behavior} "
            f"confirmed={comp.get('jailbreak_confirmed')} "
            f"gap={comp.get('loss_behavior_gap')}"
        )

### Step 6: Multi-Stage Validation

Runs transferability, perplexity, and trajectory validation for optimized suffixes. Interpret transferability as confirmed jailbreak transfer only when Step 5 shows generated compliance and behavior-gated success without prompt echo or repetition collapse.

In [ ]:
run_script('scripts/06_multi_stage_validation.py')

## 5. Backup Results to Google Drive

Copies the `outputs/` tree plus key repo files to a timestamped Google Drive folder.

In [ ]:
import os
import shutil
from datetime import datetime, timezone

drive_dest = '/content/drive/My Drive/ACT-Break-Results-gemma'

if os.path.exists('/content/drive/My Drive'):
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    run_dest = os.path.join(drive_dest, f'run_{timestamp}')
    os.makedirs(run_dest, exist_ok=True)

    print(f'Copying outputs to Google Drive: {run_dest}')
    if os.path.exists('outputs'):
        shutil.copytree('outputs', os.path.join(run_dest, 'outputs'), dirs_exist_ok=True)
        for file in ['README.md', 'config.py', 'colab/colab_notebook.ipynb']:
            if os.path.exists(file):
                dest_file = os.path.join(run_dest, file.replace('/', '_'))
                shutil.copy(file, dest_file)
        print('Successfully copied outputs to Google Drive!')
    else:
        print('No outputs directory found yet. Please run the pipeline steps first.')
else:
    print('Google Drive is not mounted. Please mount Google Drive in the setup cell first.')